In [343]:
# library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sklearn
from fifacodes import Members

In [344]:
kaggle = pd.read_csv('data/train.csv')

In [345]:
print(kaggle.shape,'\n')

for col in kaggle.columns:
    print(col)

(192, 24) 

version
team
continent
is_host
goals_scored_last_4y
goals_received_last_4y
wins_last_4y
losses_last_4y
draws_last_4y
world_cup_titles_before
squad_total_market_value_eur
fifa_rank_pre_tournament
fifa_points_pre_tournament
squad_avg_age
world_cup_participations_before
groups_passed_before
round16_before
quarterfinals_before
semifinals_before
finals_before
winner
finalist
semi_finalist
quarter_finalist


In [346]:
kaggle.iloc[:,0:12].describe()

,version,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,squad_total_market_value_eur,fifa_rank_pre_tournament
count,192.00000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,1.600000e+02,192.000000
mean,2012.00000,0.036458,84.583333,44.572917,26.046875,11.052083,11.802083,0.427083,3.433053e+08,23.541667
std,6.84916,0.187918,23.731776,13.608009,6.912534,4.745612,3.496055,1.085312,3.489032e+08,18.214497
min,2002.00000,0.000000,28.000000,19.000000,8.000000,2.000000,4.000000,0.000000,6.270000e+06,1.000000
25%,2006.00000,0.000000,66.000000,35.000000,21.000000,8.000000,10.000000,0.000000,9.385750e+07,9.000000
50%,2012.00000,0.000000,83.500000,42.500000,25.000000,10.500000,12.000000,0.000000,2.179400e+08,20.000000
75%,2018.00000,0.000000,102.000000,53.250000,31.000000,14.000000,14.000000,0.000000,4.129750e+08,34.000000
max,2022.00000,1.000000,156.000000,95.000000,48.000000,30.000000,23.000000,5.000000,1.620000e+09,105.000000


In [347]:
kaggle.iloc[:,12:].describe()

,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
count,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000
mean,954.165469,27.249635,5.625000,3.572917,2.244792,2.223958,1.348958,0.697917,0.031250,0.062500,0.125000,0.250000
std,367.544609,1.092351,5.548926,4.584848,2.701071,3.525056,2.618520,1.731263,0.174448,0.242694,0.331584,0.434145
min,285.000000,23.920000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,670.500000,26.560000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,824.000000,27.225000,4.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1182.500000,28.000000,10.000000,6.250000,4.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.250000
max,1841.300000,29.780000,21.000000,19.000000,11.000000,16.000000,13.000000,8.000000,1.000000,1.000000,1.000000,1.000000


In [348]:
kaggle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   version                          192 non-null    int64  
 1   team                             192 non-null    object 
 2   continent                        192 non-null    object 
 3   is_host                          192 non-null    int64  
 4   goals_scored_last_4y             192 non-null    int64  
 5   goals_received_last_4y           192 non-null    int64  
 6   wins_last_4y                     192 non-null    int64  
 7   losses_last_4y                   192 non-null    int64  
 8   draws_last_4y                    192 non-null    int64  
 9   world_cup_titles_before          192 non-null    int64  
 10  squad_total_market_value_eur     160 non-null    float64
 11  fifa_rank_pre_tournament         192 non-null    int64  
 12  fifa_points_pre_tourna

In [349]:
kaggle_clean = kaggle[[
    'version',
    'team',
    'is_host',
    'goals_scored_last_4y',
    'goals_received_last_4y',
    'wins_last_4y',
    'losses_last_4y',
    'draws_last_4y',
    'world_cup_titles_before',
    'fifa_rank_pre_tournament',
    'fifa_points_pre_tournament',
    'squad_avg_age',
    'world_cup_participations_before',
    'groups_passed_before',
    'round16_before',
    'quarterfinals_before',
    'semifinals_before',
    'finals_before'
]].copy()

kaggle_clean['tournament_id'] = kaggle_clean['version'].map({2002:'WC-2002',2006:'WC-2006',2010:'WC-2010',
                                                             2014:'WC-2014',2018:'WC-2018',2022:'WC-2022'})

members = Members()
fifa_dict = {member.name: member.code for name, member in members.items()}
# manually fix country codes for teams not found in fifacodes
fifa_dict['China PR'] = 'CHN'
fifa_dict['Serbia and Montenegro'] = 'SCG'
kaggle_clean['team_code'] = kaggle_clean['team'].map(fifa_dict)

kaggle_clean = kaggle_clean.drop(columns = ['version'])

In [350]:
kaggle_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   team                             192 non-null    object 
 1   is_host                          192 non-null    int64  
 2   goals_scored_last_4y             192 non-null    int64  
 3   goals_received_last_4y           192 non-null    int64  
 4   wins_last_4y                     192 non-null    int64  
 5   losses_last_4y                   192 non-null    int64  
 6   draws_last_4y                    192 non-null    int64  
 7   world_cup_titles_before          192 non-null    int64  
 8   fifa_rank_pre_tournament         192 non-null    int64  
 9   fifa_points_pre_tournament       192 non-null    float64
 10  squad_avg_age                    192 non-null    float64
 11  world_cup_participations_before  192 non-null    int64  
 12  groups_passed_before  

In [351]:
folder = 'data/data-csv/'

for file in os.listdir(folder):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(folder, file))
        print(f"\n{file}")
        print(df.shape)
        print(df.columns.tolist())


groups.csv
(159, 7)
['key_id', 'tournament_id', 'tournament_name', 'stage_number', 'stage_name', 'group_name', 'count_teams']

managers.csv
(475, 7)
['key_id', 'manager_id', 'family_name', 'given_name', 'female', 'country_name', 'manager_wikipedia_link']

qualified_teams.csv
(625, 8)
['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name', 'team_code', 'count_matches', 'performance']

teams.csv
(88, 14)
['key_id', 'team_id', 'team_name', 'team_code', 'mens_team', 'womens_team', 'federation_name', 'region_name', 'confederation_id', 'confederation_name', 'confederation_code', 'mens_team_wikipedia_link', 'womens_team_wikipedia_link', 'federation_wikipedia_link']

player_appearances.csv
(27432, 21)
['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name', 'match_date', 'stage_name', 'group_name', 'team_id', 'team_name', 'team_code', 'home_team', 'away_team', 'player_id', 'family_name', 'given_name', 'shirt_number', 'position_name', 'position_code', 'starter', 

In [352]:
# load jfjelstul tables that are most useful
matches = pd.read_csv('data/data-csv/matches.csv')
squads = pd.read_csv('data/data-csv/squads.csv')
players = pd.read_csv('data/data-csv/players.csv')
player_appearances = pd.read_csv('data/data-csv/player_appearances.csv')
goals = pd.read_csv('data/data-csv/goals.csv')
bookings = pd.read_csv('data/data-csv/bookings.csv')
subs = pd.read_csv('data/data-csv/substitutions.csv')

In [353]:
matches.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name',
       'stage_name', 'group_name', 'group_stage', 'knockout_stage', 'replayed',
       'replay', 'match_date', 'match_time', 'stadium_id', 'stadium_name',
       'city_name', 'country_name', 'home_team_id', 'home_team_name',
       'home_team_code', 'away_team_id', 'away_team_name', 'away_team_code',
       'score', 'home_team_score', 'away_team_score', 'home_team_score_margin',
       'away_team_score_margin', 'extra_time', 'penalty_shootout',
       'score_penalties', 'home_team_score_penalties',
       'away_team_score_penalties', 'result', 'home_team_win', 'away_team_win',
       'draw'],
      dtype='object')

In [354]:
# trim tables to 2002 onwards and most useful features
recent_cups = [f"WC-{4 * x + 2002}" for x in range(6)]

matches_clean = matches[matches['tournament_id'].isin(recent_cups)][[
        # identifiers
        'tournament_id',
        'match_id',
        'stage_name',
        'group_name',
        'group_stage',
        'knockout_stage',
        'match_date',

        # team info
        'home_team_id',
        'home_team_code',
        'away_team_id',
        'away_team_code',

        # scores and results
        'home_team_score',
        'away_team_score',
        'home_team_win',
        'away_team_win',
        'draw',
        'penalty_shootout'
]].copy()

In [355]:
matches_clean.head()

,tournament_id,match_id,stage_name,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,penalty_shootout
664,WC-2002,M-2002-01,group stage,Group A,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,1,0,1,0,0
665,WC-2002,M-2002-02,group stage,Group E,1,0,2002-06-01,T-60,IRL,T-11,CMR,1,1,0,0,1,0
666,WC-2002,M-2002-03,group stage,Group A,1,0,2002-06-01,T-84,URY,T-22,DNK,1,2,0,1,0,0
667,WC-2002,M-2002-04,group stage,Group E,1,0,2002-06-01,T-31,DEU,T-63,SAU,8,0,1,0,0,0
668,WC-2002,M-2002-05,group stage,Group F,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,0,1,0,0,0


In [356]:
squads.shape

(13843, 12)

In [357]:
squads.tail(10)

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code
13833,13834,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-34567,Lockyer,Tom,17,defender,DF
13834,13835,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-56664,Williams,Jonny,18,midfielder,MF
13835,13836,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-37173,Harris,Mark,19,forward,FW
13836,13837,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-74707,James,Daniel,20,forward,FW
13837,13838,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-92482,Davies,Adam,21,goal keeper,GK
13838,13839,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-20564,Thomas,Sorba,22,midfielder,MF
13839,13840,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-24982,Levitt,Dylan,23,midfielder,MF
13840,13841,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-02030,Cabango,Ben,24,defender,DF
13841,13842,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-07137,Colwill,Rubin,25,midfielder,MF
13842,13843,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-84269,Smith,Matthew,26,midfielder,MF


In [358]:
squads.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name',
       'team_code', 'player_id', 'family_name', 'given_name', 'shirt_number',
       'position_name', 'position_code'],
      dtype='object')

In [359]:
squads_clean = squads[squads['tournament_id'].isin(recent_cups)][[
    'tournament_id',
    'team_code',
    'player_id',
    'family_name',
    'given_name',
    'position_code'
]].copy()

In [360]:
squads_clean.head()

,tournament_id,team_code,player_id,family_name,given_name,position_code
7238,WC-2002,ARG,P-56385,Burgos,Germán,GK
7239,WC-2002,ARG,P-83403,Ayala,Roberto,DF
7240,WC-2002,ARG,P-60914,Sorín,Juan Pablo,DF
7241,WC-2002,ARG,P-31684,Pochettino,Mauricio,DF
7242,WC-2002,ARG,P-88203,Almeyda,Matías,MF


In [361]:
players.shape

(10401, 13)

In [362]:
players.head()

,key_id,player_id,family_name,given_name,birth_date,female,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
0,1,P-35894,A'Court,Alan,1934-09-30,0,0,0,0,1,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,2,P-29915,Aarønes,Ann Kristin,1973-01-19,1,0,0,1,1,2,"1995, 1999",https://en.wikipedia.org/wiki/Ann_Kristin_Aar%...
2,3,P-03484,Aaronson,Brenden,2000-10-22,0,0,0,0,1,1,2022,https://en.wikipedia.org/wiki/Brenden_Aaronson
3,4,P-04189,Abadzhiev,Stefan,1934-07-03,0,0,0,0,1,1,1966,https://en.wikipedia.org/wiki/Stefan_Abadzhiev
4,5,P-03523,Abalo,Jean-Paul,1975-06-26,0,0,1,0,0,1,2006,https://en.wikipedia.org/wiki/Jean-Paul_Abalo


In [363]:
players.columns

Index(['key_id', 'player_id', 'family_name', 'given_name', 'birth_date',
       'female', 'goal_keeper', 'defender', 'midfielder', 'forward',
       'count_tournaments', 'list_tournaments', 'player_wikipedia_link'],
      dtype='object')

In [364]:
players_clean = players[players['female'] == 0][[
    'player_id',
    'birth_date',
]].copy()

In [365]:
players_clean.head()

,player_id,birth_date
0,P-35894,1934-09-30
2,P-03484,2000-10-22
3,P-04189,1934-07-03
4,P-03523,1975-06-26
6,P-35138,1978-08-03


In [366]:
player_appearances.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name',
       'match_date', 'stage_name', 'group_name', 'team_id', 'team_name',
       'team_code', 'home_team', 'away_team', 'player_id', 'family_name',
       'given_name', 'shirt_number', 'position_name', 'position_code',
       'starter', 'substitute'],
      dtype='object')

In [367]:
player_appearances_clean = player_appearances[player_appearances['tournament_id'].isin(recent_cups)][[
    'tournament_id',
    'match_id',
    'team_code',
    'player_id',
    'starter',
    'substitute'
]].copy()

In [368]:
player_appearances_clean.head(20)

,tournament_id,match_id,team_code,player_id,starter,substitute
11110,WC-2002,M-2002-01,FRA,P-56735,1,0
11111,WC-2002,M-2002-01,FRA,P-96540,1,0
11112,WC-2002,M-2002-01,FRA,P-80680,1,0
11113,WC-2002,M-2002-01,FRA,P-79380,1,0
11114,WC-2002,M-2002-01,FRA,P-00916,1,0
11115,WC-2002,M-2002-01,FRA,P-51395,1,0
11116,WC-2002,M-2002-01,FRA,P-56947,1,0
11117,WC-2002,M-2002-01,FRA,P-55991,1,0
11118,WC-2002,M-2002-01,FRA,P-40400,1,0
11119,WC-2002,M-2002-01,FRA,P-19907,1,0


In [369]:
goals.columns

Index(['key_id', 'goal_id', 'tournament_id', 'tournament_name', 'match_id',
       'match_name', 'match_date', 'stage_name', 'group_name', 'team_id',
       'team_name', 'team_code', 'home_team', 'away_team', 'player_id',
       'family_name', 'given_name', 'shirt_number', 'player_team_id',
       'player_team_name', 'player_team_code', 'minute_label',
       'minute_regulation', 'minute_stoppage', 'match_period', 'own_goal',
       'penalty'],
      dtype='object')

In [370]:
goals_clean = goals[goals['tournament_id'].isin(recent_cups)][[
    'goal_id',
    'tournament_id',
    'match_id',
    'player_id',
    'player_team_code',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'own_goal',
    'penalty'
]].copy()

In [371]:
goals_clean.head()

,goal_id,tournament_id,match_id,player_id,player_team_code,minute_regulation,minute_stoppage,match_period,own_goal,penalty
2076,G-2077,WC-2002,M-2002-01,P-82212,SEN,30,0,first half,0,0
2077,G-2078,WC-2002,M-2002-02,P-18804,CMR,39,0,first half,0,0
2078,G-2079,WC-2002,M-2002-02,P-00105,IRL,52,0,second half,0,0
2079,G-2080,WC-2002,M-2002-03,P-50534,DNK,45,0,first half,0,0
2080,G-2081,WC-2002,M-2002-03,P-49428,URY,47,0,second half,0,0


In [372]:
bookings.columns

Index(['key_id', 'booking_id', 'tournament_id', 'tournament_name', 'match_id',
       'match_name', 'match_date', 'stage_name', 'group_name', 'team_id',
       'team_name', 'team_code', 'home_team', 'away_team', 'player_id',
       'family_name', 'given_name', 'shirt_number', 'minute_label',
       'minute_regulation', 'minute_stoppage', 'match_period', 'yellow_card',
       'red_card', 'second_yellow_card', 'sending_off'],
      dtype='object')

In [373]:
bookings_clean = bookings[bookings['tournament_id'].isin(recent_cups)][[
    'booking_id',
    'tournament_id',
    'match_id',
    'team_code',
    'home_team',
    'away_team',
    'player_id',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'yellow_card',
    'red_card',
    'second_yellow_card',
    'sending_off'
]].copy()

In [374]:
bookings_clean.head()

,booking_id,tournament_id,match_id,team_code,home_team,away_team,player_id,minute_regulation,minute_stoppage,match_period,yellow_card,red_card,second_yellow_card,sending_off
1216,B-1217,WC-2002,M-2002-01,FRA,1,0,P-40400,45,2,"first half, stoppage time",1,0,0,0
1217,B-1218,WC-2002,M-2002-01,SEN,0,1,P-95302,51,0,second half,1,0,0,0
1218,B-1219,WC-2002,M-2002-02,IRL,1,0,P-15705,30,0,first half,1,0,0,0
1219,B-1220,WC-2002,M-2002-02,IRL,1,0,P-56778,51,0,second half,1,0,0,0
1220,B-1221,WC-2002,M-2002-02,IRL,1,0,P-19727,82,0,second half,1,0,0,0


In [375]:
subs.columns

Index(['key_id', 'substitution_id', 'tournament_id', 'tournament_name',
       'match_id', 'match_name', 'match_date', 'stage_name', 'group_name',
       'team_id', 'team_name', 'team_code', 'home_team', 'away_team',
       'player_id', 'family_name', 'given_name', 'shirt_number',
       'minute_label', 'minute_regulation', 'minute_stoppage', 'match_period',
       'going_off', 'coming_on'],
      dtype='object')

In [376]:
subs_clean = subs[subs['tournament_id'].isin(recent_cups)][[
    'substitution_id',
    'tournament_id',
    'match_id',
    'team_code',
    'home_team',
    'away_team',
    'player_id',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'going_off',
    'coming_on'
]].copy()

In [377]:
subs_clean.head()

,substitution_id,tournament_id,match_id,team_code,home_team,away_team,player_id,minute_regulation,minute_stoppage,match_period,going_off,coming_on
3230,S-3231,WC-2002,M-2002-01,FRA,1,0,P-80680,60,0,second half,1,0
3231,S-3232,WC-2002,M-2002-01,FRA,1,0,P-74065,60,0,second half,0,1
3232,S-3233,WC-2002,M-2002-01,FRA,1,0,P-00916,81,0,second half,1,0
3233,S-3234,WC-2002,M-2002-01,FRA,1,0,P-53533,81,0,second half,0,1
3234,S-3235,WC-2002,M-2002-02,IRL,1,0,P-15705,46,0,second half,1,0


In [378]:
goals_clean.head()

,goal_id,tournament_id,match_id,player_id,player_team_code,minute_regulation,minute_stoppage,match_period,own_goal,penalty
2076,G-2077,WC-2002,M-2002-01,P-82212,SEN,30,0,first half,0,0
2077,G-2078,WC-2002,M-2002-02,P-18804,CMR,39,0,first half,0,0
2078,G-2079,WC-2002,M-2002-02,P-00105,IRL,52,0,second half,0,0
2079,G-2080,WC-2002,M-2002-03,P-50534,DNK,45,0,first half,0,0
2080,G-2081,WC-2002,M-2002-03,P-49428,URY,47,0,second half,0,0


In [379]:
goals_agg = (goals_clean
    .groupby(['tournament_id', 'match_id', 'player_team_code'])
    .agg(
        goals_scored=('goal_id', 'count'),
        own_goals=('own_goal', 'sum'),
        penalties=('penalty', 'sum')
    )
    .reset_index()
)

goals_agg.head()

,tournament_id,match_id,player_team_code,goals_scored,own_goals,penalties
0,WC-2002,M-2002-01,SEN,1,0,0
1,WC-2002,M-2002-02,CMR,1,0,0
2,WC-2002,M-2002-02,IRL,1,0,0
3,WC-2002,M-2002-03,DNK,2,0,0
4,WC-2002,M-2002-03,URY,1,0,0


In [407]:
match_table = (matches_clean
    .merge(
        goals_agg.rename(columns = {'player_team_code':'home_team_code', 'goals_scored': 'home_goals_scored',
                                  'own_goals':'home_own_goals', 'penalties': 'home_penalties'}),
        on = ['tournament_id','match_id','home_team_code'], how = 'left'
    )
    .merge(
        goals_agg.rename(columns = {'player_team_code':'away_team_code', 'goals_scored':'away_goals_scored',
                                  'own_goals':'away_own_goals', 'penalties':'away_penalties'}),
        on = ['tournament_id','match_id','away_team_code'], how = 'left'
    )
    .fillna(0)
)

match_table['group_name'] = match_table['group_name'].str[6:]

In [381]:
match_table.head()

,tournament_id,match_id,stage_name,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,...,home_team_win,away_team_win,draw,penalty_shootout,home_goals_scored,home_own_goals,home_penalties,away_goals_scored,away_own_goals,away_penalties
0,WC-2002,M-2002-01,group stage,A,1,0,2002-05-31,T-30,FRA,T-65,...,0,1,0,0,0.0,0.0,0.0,1.0,0.0,0.0
1,WC-2002,M-2002-02,group stage,E,1,0,2002-06-01,T-60,IRL,T-11,...,0,0,1,0,1.0,0.0,0.0,1.0,0.0,0.0
2,WC-2002,M-2002-03,group stage,A,1,0,2002-06-01,T-84,URY,T-22,...,0,1,0,0,1.0,0.0,0.0,2.0,0.0,0.0
3,WC-2002,M-2002-04,group stage,E,1,0,2002-06-01,T-31,DEU,T-63,...,1,0,0,0,8.0,0.0,0.0,0.0,0.0,0.0
4,WC-2002,M-2002-05,group stage,F,1,0,2002-06-02,T-03,ARG,T-50,...,1,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0


In [408]:
match_table = (match_table
    .merge(
        kaggle_clean[['tournament_id', 'team_code', 'is_host', 'goals_scored_last_4y', 'goals_received_last_4y',
                      'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'fifa_rank_pre_tournament',
                      'fifa_points_pre_tournament']]
        .rename(columns={
            'team_code': 'home_team_code',
            'is_host': 'home_is_host',
            'goals_scored_last_4y': 'home_goals_scored_last_4y',
            'goals_received_last_4y': 'home_goals_received_last_4y',
            'wins_last_4y': 'home_wins_last_4y',
            'losses_last_4y': 'home_losses_last_4y',
            'draws_last_4y': 'home_draws_last_4y',
            'fifa_rank_pre_tournament': 'home_fifa_rank_pre_tournament',
            'fifa_points_pre_tournament': 'home_fifa_points_pre_tournament'
        }),
        on=['tournament_id', 'home_team_code'], how='left'
    )
    .merge(
        kaggle_clean[['tournament_id', 'team_code', 'is_host', 'goals_scored_last_4y', 'goals_received_last_4y',
                      'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'fifa_rank_pre_tournament',
                      'fifa_points_pre_tournament']]
        .rename(columns={
            'team_code': 'away_team_code',
            'is_host': 'away_is_host',
            'goals_scored_last_4y': 'away_goals_scored_last_4y',
            'goals_received_last_4y': 'away_goals_received_last_4y',
            'wins_last_4y': 'away_wins_last_4y',
            'losses_last_4y': 'away_losses_last_4y',
            'draws_last_4y': 'away_draws_last_4y',
            'fifa_rank_pre_tournament': 'away_fifa_rank_pre_tournament',
            'fifa_points_pre_tournament': 'away_fifa_points_pre_tournament'
        }),
        on=['tournament_id', 'away_team_code'], how='left'
    )
)

In [409]:
# drop columns that would create data leakage for training
match_table = match_table.drop(columns = ['stage_name','home_own_goals','home_penalties','away_own_goals',
                                          'away_penalties','penalty_shootout'])

# convert match date to year
match_table['match_date'] = pd.to_datetime(match_table['match_date'])
# calculate difference in fifa rank and points between teams
match_table['fifa_rank_diff'] = match_table['home_fifa_rank_pre_tournament'] - match_table['away_fifa_rank_pre_tournament']
match_table['fifa_pt_diff'] = match_table['home_fifa_points_pre_tournament'] - match_table['away_fifa_points_pre_tournament']

drop_before_training = ['tournament_id','match_id','home_team_id','home_team_code','away_team_id','away_team_code',
                        'match_date','home_team_win','draw','away_team_win','group_name','home_goals_scored',
                        'away_goals_scored','home_team_score','away_team_score']

In [384]:
match_table.head()

,tournament_id,match_id,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_win,...,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff
0,WC-2002,M-2002-01,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,...,0.0,79.0,46.0,26.0,12.0,21.0,42.0,599.0,-41.0,203.0
1,WC-2002,M-2002-02,1,0,2002-06-01,T-60,IRL,T-11,CMR,0,...,0.0,65.0,25.0,23.0,5.0,13.0,17.0,672.0,-2.0,2.0
2,WC-2002,M-2002-03,1,0,2002-06-01,T-84,URY,T-22,DNK,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,WC-2002,M-2002-04,1,0,2002-06-01,T-31,DEU,T-63,SAU,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,WC-2002,M-2002-05,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,...,0.0,71.0,36.0,26.0,7.0,12.0,27.0,644.0,-24.0,140.0


In [410]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'fifa_rank_diff', 'fifa_pt_diff'],
      dtype='object')

In [411]:
match_table[match_table['home_fifa_rank_pre_tournament'].isna()]['tournament_id'].value_counts()


tournament_id
WC-2010    29
WC-2006    23
WC-2014    21
WC-2002    19
WC-2022    17
WC-2018    16
Name: count, dtype: int64

In [412]:
unmatched = match_table[match_table['home_fifa_rank_pre_tournament'].isna()]['home_team_code'].unique()
print(sorted(unmatched))

['AGO', 'CHE', 'CHL', 'CRI', 'DEU', 'DNK', 'DZA', 'GRC', 'HND', 'HRV', 'NLD', 'PRT', 'PRY', 'SAU', 'TGO', 'TTO', 'URY', 'ZAF']


In [413]:
player_appearances_clean.merge(players_clean, on = 'player_id').head()

,tournament_id,match_id,team_code,player_id,starter,substitute,birth_date
0,WC-2002,M-2002-01,FRA,P-56735,1,0,1969-12-09
1,WC-2002,M-2002-01,FRA,P-96540,1,0,1976-06-23
2,WC-2002,M-2002-01,FRA,P-80680,1,0,1968-03-09
3,WC-2002,M-2002-01,FRA,P-79380,1,0,1968-09-07
4,WC-2002,M-2002-01,FRA,P-00916,1,0,1974-05-10


In [ ]:
code_map = {
    'DEU': 'GER',  # Germany
    'CHE': 'SUI',  # Switzerland
    'HRV': 'CRO',  # Croatia
    'NLD': 'NED',  # Netherlands
    'PRT': 'POR',  # Portugal
    'DZA': 'ALG',  # Algeria
    'CHL': 'CHI',  # Chile
    'DNK': 'DEN',  # Denmark
    'GRC': 'GRE',  # Greece
    'PRY': 'PAR',  # Paraguay
    'SAU': 'KSA',  # Saudi Arabia
    'URY': 'URU',  # Uruguay
    'ZAF': 'RSA',  # South Africa
}

In [432]:
for df in [match_table, kaggle_clean]:
    for col in ['home_team_code', 'away_team_code', 'team_code']:
        if col in df.columns:
            df[col] = df[col].replace(code_map)

In [389]:
# calculate average age of each starting 11

# start with player_appearances_clean table, where only starters are included
squad_avg_age = (player_appearances_clean[player_appearances_clean['starter'] == 1]
    # add birthdates 
    .merge(players_clean, on = 'player_id', how = 'left')
    # add match_date
    .merge(matches_clean[['tournament_id', 'match_id', 'match_date']], on = ['tournament_id', 'match_id'], how = 'left')
    # calculate age by subtracting birthdate
    .assign(age=lambda df: (pd.to_datetime(df['match_date']) - pd.to_datetime(df['birth_date'])).dt.days // 365)
    # find the mean age for players in each team for each match
    .groupby(['tournament_id', 'match_id', 'team_code'])['age']
    .mean()
    .reset_index()
    .rename(columns = {'age': 'squad_avg_age'})
)

In [390]:
squad_avg_age.head()

,tournament_id,match_id,team_code,squad_avg_age
0,WC-2002,M-2002-01,FRA,29.545455
1,WC-2002,M-2002-01,SEN,25.363636
2,WC-2002,M-2002-02,CMR,25.454545
3,WC-2002,M-2002-02,IRL,26.727273
4,WC-2002,M-2002-03,DNK,28.000000


In [391]:
match_table.head()

,tournament_id,match_id,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_win,...,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff
0,WC-2002,M-2002-01,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,...,0.0,79.0,46.0,26.0,12.0,21.0,42.0,599.0,-41.0,203.0
1,WC-2002,M-2002-02,1,0,2002-06-01,T-60,IRL,T-11,CMR,0,...,0.0,65.0,25.0,23.0,5.0,13.0,17.0,672.0,-2.0,2.0
2,WC-2002,M-2002-03,1,0,2002-06-01,T-84,URY,T-22,DNK,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,WC-2002,M-2002-04,1,0,2002-06-01,T-31,DEU,T-63,SAU,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,WC-2002,M-2002-05,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,...,0.0,71.0,36.0,26.0,7.0,12.0,27.0,644.0,-24.0,140.0


In [414]:
match_table = (match_table
    .merge(
        squad_avg_age.rename(columns = {
            'team_code': 'home_team_code',
            'squad_avg_age': 'home_squad_avg_age'
        }),
        on = ['tournament_id','match_id', 'home_team_code'], how = 'left'
    )
    .merge(
        squad_avg_age.rename(columns = {
            'team_code': 'away_team_code',
            'squad_avg_age': 'away_squad_avg_age'
        }),
        on = ['tournament_id','match_id', 'away_team_code'], how = 'left'
    )
)

# calculate squad age difference
match_table['squad_age_diff'] = match_table['home_squad_avg_age'] - match_table['away_squad_avg_age']

In [393]:
match_table.head()

,tournament_id,match_id,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_win,...,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff
0,WC-2002,M-2002-01,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,...,26.0,12.0,21.0,42.0,599.0,-41.0,203.0,29.545455,25.363636,4.181818
1,WC-2002,M-2002-02,1,0,2002-06-01,T-60,IRL,T-11,CMR,0,...,23.0,5.0,13.0,17.0,672.0,-2.0,2.0,26.727273,25.454545,1.272727
2,WC-2002,M-2002-03,1,0,2002-06-01,T-84,URY,T-22,DNK,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.181818,28.000000,-1.818182
3,WC-2002,M-2002-04,1,0,2002-06-01,T-31,DEU,T-63,SAU,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.181818,26.727273,0.454545
4,WC-2002,M-2002-05,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,...,26.0,7.0,12.0,27.0,644.0,-24.0,140.0,28.000000,25.181818,2.818182


In [415]:
home = match_table[match_table['group_stage'] == 1][
    ['tournament_id', 'match_id', 'match_date', 'home_team_code', 'home_team_win', 'draw']
].rename(columns={
    'home_team_code': 'team_code',
    'home_team_win': 'win'
})

away = match_table[match_table['group_stage'] == 1][
    ['tournament_id', 'match_id', 'match_date', 'away_team_code', 'away_team_win', 'draw']
].rename(columns={
    'away_team_code': 'team_code',
    'away_team_win': 'win'
})

team_group_matches = pd.concat([home, away]).sort_values(['tournament_id', 'team_code', 'match_date'])

team_group_matches['points'] = team_group_matches['win'] * 3 + team_group_matches['draw'] * 1

In [416]:
team_group_matches['cumulative_points'] = (team_group_matches
    .groupby(['tournament_id', 'team_code'])['points']
    .transform(lambda x: x.shift(1).cumsum())
)

In [417]:
match_table = (match_table
    .merge(
        team_group_matches[['tournament_id', 'match_id', 'team_code', 'cumulative_points']]
        .rename(columns={
            'team_code': 'home_team_code',
            'cumulative_points': 'home_group_points'
        }),
        on=['tournament_id', 'match_id', 'home_team_code'], how='left'
    )
    .merge(
        team_group_matches[['tournament_id', 'match_id', 'team_code', 'cumulative_points']]
        .rename(columns={
            'team_code': 'away_team_code',
            'cumulative_points': 'away_group_points'
        }),
        on=['tournament_id', 'match_id', 'away_team_code'], how='left'
    )
)

In [397]:
match_table.tail(10)

,tournament_id,match_id,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_win,...,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points
374,WC-2022,M-2022-55,0,1,2022-12-06,T-47,MAR,T-73,ESP,1,...,14.0,7.0,1715.22,15.0,-151.72,26.818182,25.727273,1.090909,NaN,NaN
375,WC-2022,M-2022-56,0,1,2022-12-06,T-58,PRT,T-75,CHE,1,...,NaN,NaN,NaN,NaN,NaN,26.818182,28.272727,-1.454545,NaN,NaN
376,WC-2022,M-2022-57,0,1,2022-12-09,T-18,HRV,T-09,BRA,1,...,10.0,1.0,1841.30,NaN,NaN,28.818182,28.090909,0.727273,NaN,NaN
377,WC-2022,M-2022-58,0,1,2022-12-09,T-48,NLD,T-03,ARG,0,...,13.0,3.0,1773.88,NaN,NaN,27.000000,26.909091,0.090909,NaN,NaN
378,WC-2022,M-2022-59,0,1,2022-12-10,T-47,MAR,T-58,PRT,1,...,NaN,NaN,NaN,NaN,NaN,27.363636,26.363636,1.000000,NaN,NaN
379,WC-2022,M-2022-60,0,1,2022-12-10,T-28,ENG,T-30,FRA,0,...,12.0,4.0,1759.78,1.0,-31.31,26.363636,27.363636,-1.000000,NaN,NaN
380,WC-2022,M-2022-61,0,1,2022-12-13,T-03,ARG,T-18,HRV,1,...,NaN,NaN,NaN,NaN,NaN,27.181818,28.818182,-1.636364,NaN,NaN
381,WC-2022,M-2022-62,0,1,2022-12-14,T-30,FRA,T-47,MAR,1,...,12.0,22.0,1563.50,-18.0,196.28,27.000000,26.909091,0.090909,NaN,NaN
382,WC-2022,M-2022-63,0,1,2022-12-17,T-18,HRV,T-47,MAR,1,...,12.0,22.0,1563.50,NaN,NaN,27.454545,26.181818,1.272727,NaN,NaN
383,WC-2022,M-2022-64,0,1,2022-12-18,T-03,ARG,T-30,FRA,1,...,12.0,4.0,1759.78,-1.0,14.10,27.818182,27.545455,0.272727,NaN,NaN


In [418]:
final_group_points = (team_group_matches
    .groupby(['tournament_id', 'team_code'])['points']
    .sum()
    .reset_index()
    .rename(columns={'points': 'final_group_points'})
)

In [419]:
match_table = (match_table
    .merge(
        final_group_points.rename(columns={
            'team_code': 'home_team_code',
            'final_group_points': 'home_final_group_points'
        }),
        on=['tournament_id', 'home_team_code'], how='left'
    )
    .merge(
        final_group_points.rename(columns={
            'team_code': 'away_team_code',
            'final_group_points': 'away_final_group_points'
        }),
        on=['tournament_id', 'away_team_code'], how='left'
    )
)

In [420]:
match_table['home_group_points'] = match_table['home_group_points'].fillna(
    match_table['home_final_group_points']
)
match_table['away_group_points'] = match_table['away_group_points'].fillna(
    match_table['away_final_group_points']
)
match_table = match_table.drop(columns=['home_final_group_points', 'away_final_group_points'])

In [421]:
match_table['group_pts_diff'] = match_table['home_group_points'] - match_table['away_group_points'] 

In [422]:
match_table.tail()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,...,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points,group_pts_diff
379,WC-2022,M-2022-60,plicable,0,1,2022-12-10,T-28,ENG,T-30,FRA,...,4.0,1759.78,1.0,-31.31,26.363636,27.363636,-1.000000,7.0,6.0,1.0
380,WC-2022,M-2022-61,plicable,0,1,2022-12-13,T-03,ARG,T-18,HRV,...,NaN,NaN,NaN,NaN,27.181818,28.818182,-1.636364,6.0,5.0,1.0
381,WC-2022,M-2022-62,plicable,0,1,2022-12-14,T-30,FRA,T-47,MAR,...,22.0,1563.50,-18.0,196.28,27.000000,26.909091,0.090909,6.0,7.0,-1.0
382,WC-2022,M-2022-63,plicable,0,1,2022-12-17,T-18,HRV,T-47,MAR,...,22.0,1563.50,NaN,NaN,27.454545,26.181818,1.272727,5.0,7.0,-2.0
383,WC-2022,M-2022-64,plicable,0,1,2022-12-18,T-03,ARG,T-30,FRA,...,4.0,1759.78,-1.0,14.10,27.818182,27.545455,0.272727,6.0,6.0,0.0


In [405]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_stage', 'knockout_stage',
       'match_date', 'home_team_id', 'home_team_code', 'away_team_id',
       'away_team_code', 'home_team_win', 'away_team_win', 'draw',
       'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'fifa_rank_diff', 'fifa_pt_diff', 'home_squad_avg_age',
       'away_squad_avg_age', 'squad_age_diff', 'home_group_points',
       'away_group_points', 'group_pts_diff'],
      dtype='object')

In [423]:

home = match_table[['tournament_id', 'match_id', 'match_date', 'home_team_code', 'home_team_score', 'away_team_score']].rename(columns={
    'home_team_code': 'team_code',
    'home_team_score': 'goals_scored',
    'away_team_score': 'goals_conceded'
})

away = match_table[['tournament_id', 'match_id', 'match_date', 'away_team_code', 'away_team_score', 'home_team_score']].rename(columns={
    'away_team_code': 'team_code',
    'away_team_score': 'goals_scored',
    'home_team_score': 'goals_conceded'
})

team_match_goals = pd.concat([home, away]).sort_values(['tournament_id', 'team_code', 'match_date'])

In [424]:
team_match_goals['rolling_avg_goals_scored'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_scored']
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_match_goals['rolling_avg_goals_conceded'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_conceded']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [425]:
team_match_goals['rolling_avg_goals_scored'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_scored']
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_match_goals['rolling_avg_goals_conceded'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_conceded']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [426]:
match_table = (match_table
    .merge(
        team_match_goals[['tournament_id', 'match_id', 'team_code', 
                          'rolling_avg_goals_scored', 'rolling_avg_goals_conceded']]
        .rename(columns={
            'team_code': 'home_team_code',
            'rolling_avg_goals_scored': 'home_rolling_avg_goals_scored',
            'rolling_avg_goals_conceded': 'home_rolling_avg_goals_conceded'
        }),
        on=['tournament_id', 'match_id', 'home_team_code'], how='left'
    )
    .merge(
        team_match_goals[['tournament_id', 'match_id', 'team_code',
                          'rolling_avg_goals_scored', 'rolling_avg_goals_conceded']]
        .rename(columns={
            'team_code': 'away_team_code',
            'rolling_avg_goals_scored': 'away_rolling_avg_goals_scored',
            'rolling_avg_goals_conceded': 'away_rolling_avg_goals_conceded'
        }),
        on=['tournament_id', 'match_id', 'away_team_code'], how='left'
    )
)

In [428]:
match_table.tail()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,...,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points,group_pts_diff,home_rolling_avg_goals_scored,home_rolling_avg_goals_conceded,away_rolling_avg_goals_scored,away_rolling_avg_goals_conceded
379,WC-2022,M-2022-60,plicable,0,1,2022-12-10,T-28,ENG,T-30,FRA,...,26.363636,27.363636,-1.000000,7.0,6.0,1.0,3.0,0.500000,2.250000,1.000000
380,WC-2022,M-2022-61,plicable,0,1,2022-12-13,T-03,ARG,T-18,HRV,...,27.181818,28.818182,-1.636364,6.0,5.0,1.0,1.8,1.000000,1.200000,0.600000
381,WC-2022,M-2022-62,plicable,0,1,2022-12-14,T-30,FRA,T-47,MAR,...,27.000000,26.909091,0.090909,6.0,7.0,-1.0,2.2,1.000000,1.000000,0.200000
382,WC-2022,M-2022-63,plicable,0,1,2022-12-17,T-18,HRV,T-47,MAR,...,27.454545,26.181818,1.272727,5.0,7.0,-2.0,1.0,1.000000,0.833333,0.500000
383,WC-2022,M-2022-64,plicable,0,1,2022-12-18,T-03,ARG,T-30,FRA,...,27.818182,27.545455,0.272727,6.0,6.0,0.0,2.0,0.833333,2.166667,0.833333


In [429]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'fifa_rank_diff', 'fifa_pt_diff', 'home_squad_avg_age',
       'away_squad_avg_age', 'squad_age_diff', 'home_group_points',
       'away_group_points', 'group_pts_diff', 'home_rolling_avg_goals_scored',
      

In [430]:
match_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   tournament_id                    384 non-null    object        
 1   match_id                         384 non-null    object        
 2   group_name                       384 non-null    object        
 3   group_stage                      384 non-null    int64         
 4   knockout_stage                   384 non-null    int64         
 5   match_date                       384 non-null    datetime64[ns]
 6   home_team_id                     384 non-null    object        
 7   home_team_code                   384 non-null    object        
 8   away_team_id                     384 non-null    object        
 9   away_team_code                   384 non-null    object        
 10  home_team_score                  384 non-null    int64        